# 2장 (실습 1): 컨테이너 격리의 벽과 그 벽을 무너뜨리는 설정 확인해보기

> 책 『AI 보안, 클라우드에서 시작한다』 2장 [실습] 1번을 노트북에서 단계별로 실행하는 버전입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다. 소요 시간은 30~40분입니다.

## 🎯 학습 목표
1. 컨테이너가 VM이 아니라 **호스트 커널 위의 평범한 프로세스**이며, 네임스페이스가 "볼 수 있는 범위"만 나눈다는 점을 직접 확인합니다.
2. `-v /:/host`, `--pid=host`, `--privileged` 같은 **설정 한 줄이 어떤 격리의 벽을 없애는지** 숫자로 확인합니다.
3. 같은 위험 설정을 쿠버네티스 Pod 설정 파일(YAML)에서 **배포 전에 찾고(Trivy)**, 클러스터가 **배포 단계에서 막게(Pod Security Admission)** 만듭니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | 예정 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | 예정 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | 예정 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | 예정 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | 예정 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

## 실습 시나리오

개발팀은 "컨테이너는 VM처럼 따로 떨어져 있으니 안전하다"고 생각합니다. 급한 장애가 나면 "권한 문제 같으니 일단 다 열어 보자"며 `--privileged`나 호스트 디렉터리 마운트를 넣기도 합니다. 이 실습에서는 그런 설정 한 줄이 격리를 어떻게 없애는지 직접 확인합니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Windows PC라면 [8장 README](../ch08-devsecops-image-gate/README.md)의 [실습 준비]에서 WSL2·Docker·Trivy 설치를 먼저 마칩니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> ```bash
> sudo apt install -y python3-venv          # venv 모듈이 없을 때만
> python3 -m venv ~/.venv-lab
> ~/.venv-lab/bin/pip install jupyterlab
> cd <이 저장소 경로>/ch02-container-security
> ~/.venv-lab/bin/jupyter lab               # 표시되는 http://localhost:8888/... 주소를 브라우저로 엽니다
> ```
> - VS Code를 쓴다면 WSL 확장으로 이 폴더를 연 뒤, 노트북 오른쪽 위 'Select Kernel'에서 `~/.venv-lab`의 Python을 고릅니다.
> - 노트북은 **반드시 `ch02-container-security` 폴더에서 열어야** 합니다. 실습 파일 `k8s/bad-pod.yaml`, `k8s/good-pod.yaml`을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Trivy 0.76.0, kind 0.24.0(쿠버네티스 1.31)에서 실행한 결과입니다. 프로세스 번호, 호스트명 같은 값은 실행할 때마다 다릅니다.

> ⚠️ **주의**
> 3단계에서는 컨테이너 안에서 **호스트에 파일을 만듭니다**. 실습용 PC나 WSL에서만 하고, 회사 서버나 운영 환경에서는 하지 마세요.

## 0. 실습 준비: 도구와 실습 파일 확인

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | 예정 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | 예정 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | 예정 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | 예정 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

노트북 전체에서 쓸 도우미 함수 `sh()`를 만들고, 필요한 도구가 설치되어 있는지 확인합니다.

- `docker`, `trivy`는 **필수**입니다.
- `kind`, `kubectl`은 **5단계(선택)** 에서만 씁니다. 없으면 5단계에서 설치합니다. 5단계 안내대로 `~/bin`에 설치하면 이 셀이 `PATH`에 추가해 줍니다.

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 ch02-container-security 폴더
K8S = LAB / "k8s"         # 실습 파일 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 5단계에서 kind·kubectl을 ~/bin에 둔다


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


VERSION_CMD = {
    "docker": "docker --version",
    "trivy": "trivy --version | head -1",
    "kind": "kind --version",
    "kubectl": "kubectl version --client | head -1",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd)}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:8} {state}")

assert K8S.exists(), "k8s 폴더가 보이지 않습니다. 노트북을 ch02-container-security 폴더에서 열었는지 확인하세요."
print("\n실습 파일:", sorted(p.name for p in K8S.glob("*.yaml")))

docker   ✅ Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2
trivy    ✅ Version: 0.76.0
kind     ✅ kind version 0.24.0


kubectl  ✅ Client Version: v1.31.1

실습 파일: ['bad-pod.yaml', 'good-pod.yaml']


Docker를 `sudo` 없이 부를 수 있는지 확인합니다. `permission denied`가 나오면 WSL 터미널에서 `sudo usermod -aG docker $USER`를 실행하고, WSL을 다시 시작한 뒤(`wsl --shutdown`) Jupyter를 다시 엽니다.

In [2]:
%%bash
docker info --format 'Docker 서버 {{.ServerVersion}} / 실행 중인 컨테이너 {{.ContainersRunning}}개'

Docker 서버 29.1.3 / 실행 중인 컨테이너 0개


## 1. 컨테이너 두 개 띄우고 격리 확인하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | 예정 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | 예정 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | 예정 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 1.1. 컨테이너 두 개 실행

버전을 고정한 경량 nginx 이미지로 컨테이너 두 개를 띄웁니다. 처음 실행하면 이미지를 내려받느라 조금 걸립니다.

> 📌 (참고) 노트북을 다시 실행할 때 같은 이름의 컨테이너가 남아 있으면 오류가 나므로, 먼저 지우고 시작합니다.

In [3]:
%%bash
docker rm -f box-a box-b >/dev/null 2>&1
docker pull -q nginx:1.27-alpine      # 3단계에서 쓸 alpine 이미지도 함께 받아 둡니다
docker pull -q alpine:3.20
docker run -d --name box-a nginx:1.27-alpine
docker run -d --name box-b nginx:1.27-alpine

docker.io/library/nginx:1.27-alpine


docker.io/library/alpine:3.20


b410f31fb3504f44963ab7f60570c789c2547d7da1fb1f8489521bf6e562a54f


6e9d38ae820bdd3318a4c52aa4d1c0986ae646b3bd17e6e5a1ca977ddcdadbb9


### 1.2. 컨테이너 안에서 본 모습

컨테이너 안에서 보이는 프로세스와 호스트명을 확인합니다.

In [4]:
%%bash
echo "── box-a 안의 프로세스"
docker exec box-a ps
echo "── 호스트명"
echo "box-a: $(docker exec box-a hostname)"
echo "box-b: $(docker exec box-b hostname)"

── box-a 안의 프로세스


PID   USER     TIME  COMMAND
    1 root      0:00 nginx: master process nginx -g daemon off;
   30 n

ginx     0:00 nginx: worker process
   31 nginx     0:00 nginx: worker process
   32 nginx     0:00 

nginx: worker process
   33 nginx     0:00 nginx: worker process
   34 nginx     0:00 nginx: worker 

process
   35 nginx     0:00 nginx: worker process
   36 nginx     0:00 nginx: worker process
   37 

nginx     0:00 nginx: worker process
   38 root      0:00 ps


── 호스트명


box-a: b410f31fb350


box-b: 6e9d38ae820b


box-a 안에서는 nginx가 **1번 프로세스(PID 1)** 이고, box-b의 프로세스는 보이지 않습니다. 호스트명도 컨테이너마다 다릅니다. PID 네임스페이스와 UTS 네임스페이스가 "볼 수 있는 범위"를 나눈 결과입니다.

### 1.3. 호스트에서 본 모습

이번에는 같은 프로세스를 **호스트에서** 봅니다.

In [5]:
%%bash
echo "── 각 컨테이너의 PID 1이 호스트에서 가진 번호"
docker inspect -f '{{.Name}} → {{.State.Pid}}' box-a box-b
echo "── 호스트의 프로세스 목록에서 찾기"
ps -eo pid,user,cmd | grep "nginx: master" | grep -v grep

── 각 컨테이너의 PID 1이 호스트에서 가진 번호


/box-a → 936
/box-b → 1041


── 호스트의 프로세스 목록에서 찾기


    936 root     nginx: master process nginx -g daemon off;
   1041 root     nginx: master process n

ginx -g daemon off;


컨테이너 안에서 1번이던 프로세스가 호스트에서는 수백, 수천 번대의 평범한 프로세스입니다. 그림 2-2 아래쪽의 "호스트에서 본 모습"이 바로 이것입니다.

### 1.4. 네임스페이스 번호 비교하기

리눅스는 프로세스가 속한 네임스페이스를 `/proc/<PID>/ns/` 아래의 번호로 보여 줍니다. 번호가 같으면 같은 "방"에 있는 것이고, 다르면 서로 다른 방에 있는 것입니다.

In [6]:
%%bash
printf '%-10s %-25s %s\n' "위치" "PID 네임스페이스" "UTS 네임스페이스"
printf '%-10s %-25s %s\n' "호스트" "$(readlink /proc/self/ns/pid)" "$(readlink /proc/self/ns/uts)"
for c in box-a box-b; do
  printf '%-10s %-25s %s\n' "$c" "$(docker exec $c readlink /proc/self/ns/pid)" "$(docker exec $c readlink /proc/self/ns/uts)"
done

위치     PID 네임스페이스    UTS 네임스페이스


호스트  pid:[4026532214]          uts:[4026532212]


box-a      pid:[4026532226]          uts:[4026532224]


box-b      pid:[4026532297]          uts:[4026532295]


호스트, box-a, box-b의 번호가 모두 다릅니다. 세 프로세스는 **같은 커널** 위에서 돌지만 서로 다른 PID·UTS 네임스페이스에 들어 있습니다.

> 🙋 (생각해 보기)
> 커널이 하나라는 것은 무엇을 뜻할까요? 커널에 취약점이 있으면 모든 컨테이너가 함께 영향을 받습니다. 2-2의 "컨테이너 탈출"이 VM 탈출보다 쉬운 이유입니다.

## 2. 컨테이너 안의 root는 호스트에서 누구인가

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | ✅ 완료 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 설정 한 줄로 벽 무너뜨리기 | 예정 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | 예정 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

In [7]:
%%bash
docker exec box-a id

uid=0(root) gid=0(root) groups=0(root),0(root),1(bin),2(daemon),3(sys),4(adm),6(disk),10(wheel),11(f

loppy),20(dialout),26(tape),27(video)


컨테이너 안의 사용자는 root(UID 0)입니다. 1.3의 호스트 `ps`에서 본 USER 열도 **root**였습니다.

이를 네임스페이스로도 확인해 봅니다. User 네임스페이스 번호가 호스트와 같으면, 컨테이너 안의 UID가 호스트의 UID와 그대로 같다는 뜻입니다.

In [8]:
%%bash
echo "호스트 : $(readlink /proc/self/ns/user)"
echo "box-a  : $(docker exec box-a readlink /proc/self/ns/user)"

호스트 : user:[4026531837]


box-a  : user:[4026531837]


1.4의 PID·UTS와 달리 **User 네임스페이스 번호는 호스트와 같습니다**. 도커는 기본 설정에서 User 네임스페이스를 쓰지 않으므로(2-1), 컨테이너 안의 root가 곧 호스트의 root입니다. 지금은 다른 네임스페이스의 벽이 막아 주고 있을 뿐입니다. 다음 단계에서 그 벽을 하나씩 걷어 봅니다.

## 3. 설정 한 줄로 벽 무너뜨리기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | ✅ 완료 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | ✅ 완료 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | 예정 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

### 3.1. 호스트의 루트 경로 마운트 (`-v /:/host`)

호스트의 `/` 전체를 컨테이너 안의 `/host`로 붙인 뒤, 컨테이너 안에서 호스트의 `/tmp`에 파일을 만듭니다.

In [9]:
%%bash
docker run --rm -v /:/host alpine:3.20 sh -c 'id; touch /host/tmp/escaped-from-container'
echo "── 호스트에서 확인"
ls -l /tmp/escaped-from-container

uid=0(root) gid=0(root) groups=0(root),0(root),1(bin),2(daemon),3(sys),4(adm),6(disk),10(wheel),11(f

loppy),20(dialout),26(tape),27(video)


── 호스트에서 확인


-rw-r--r-- 1 root root 0 Oct 23 00:01 /tmp/escaped-from-container


컨테이너 안에서 만든 파일이 **호스트의 /tmp에 root 소유로** 생겼습니다. 같은 방법으로 호스트의 설정 파일, 다른 컨테이너의 데이터, 인증서도 읽고 바꿀 수 있습니다. 컨테이너 안의 root와 호스트 경로 마운트가 만나면, 컨테이너는 사실상 호스트 그 자체가 됩니다.

같은 마운트라도 **일반 사용자(UID 1000)로 실행**하면 결과가 달라집니다.

In [10]:
%%bash
docker run --rm --user 1000 -v /:/host alpine:3.20 sh -c 'id; touch /host/root/escaped' \
  || echo "→ 쓰기 실패 (예상한 결과)"

uid=1000 gid=0(root) groups=0(root)


touch: /host/root/escaped: Permission denied


→ 쓰기 실패 (예상한 결과)


root로 실행하지 않는 것만으로도 피해 범위가 크게 줄어듭니다. 그래도 일반 사용자가 읽을 수 있는 호스트 파일은 여전히 보이므로, 근본 대책은 **필요 없는 호스트 경로를 마운트하지 않는 것**입니다.

### 3.2. 호스트 프로세스 공유 (`--pid=host`)와 장치 권한 (`--privileged`)

이번에는 옵션이 있을 때와 없을 때를 파이썬으로 나란히 세어 표로 만듭니다.

- `ps | wc -l`: 컨테이너 안에서 보이는 프로세스 목록의 줄 수
- `ls /dev | wc -l`: 컨테이너 안에서 보이는 장치 파일 수
- `readlink /proc/self/ns/pid`: 1.4에서 본 PID 네임스페이스 번호

In [11]:
def in_alpine(opts, cmd):
    """alpine 컨테이너를 opts 옵션으로 띄워 cmd를 실행한 결과를 돌려줍니다."""
    return sh(f"docker run --rm {opts} alpine:3.20 {cmd}")


def count_lines(opts, cmd):
    return len(in_alpine(opts, cmd).splitlines())


host_pid_ns = sh("readlink /proc/self/ns/pid")
rows = [
    ("`--pid=host`", "보이는 프로세스 줄 수",
     count_lines("", "ps"), count_lines("--pid=host", "ps")),
    ("`--pid=host`", "PID 네임스페이스가 호스트와 같은가",
     in_alpine("", "readlink /proc/self/ns/pid") == host_pid_ns,
     in_alpine("--pid=host", "readlink /proc/self/ns/pid") == host_pid_ns),
    ("`--privileged`", "/dev 장치 항목 수",
     count_lines("", "ls /dev"), count_lines("--privileged", "ls /dev")),
]

table = ["| 옵션 | 측정 항목 | 옵션 없음 | 옵션 있음 |", "|---|---|---|---|"]
table += [f"| {o} | {m} | {a} | {b} |" for o, m, a, b in rows]
display(Markdown("\n".join(table)))

| 옵션 | 측정 항목 | 옵션 없음 | 옵션 있음 |
|---|---|---|---|
| `--pid=host` | 보이는 프로세스 줄 수 | 2 | 69 |
| `--pid=host` | PID 네임스페이스가 호스트와 같은가 | False | True |
| `--privileged` | /dev 장치 항목 수 | 15 | 180 |

- 기본 설정에서는 머리글과 자기 자신(`ps`) 두 줄만 보이지만, `--pid=host`를 붙이면 호스트와 다른 컨테이너의 프로세스가 모두 보입니다. PID 네임스페이스 번호도 호스트와 같아집니다. **PID 네임스페이스의 벽이 사라진 것**입니다.
- `--privileged` 컨테이너에는 호스트의 디스크와 장치가 거의 모두 보입니다. 장치에 직접 접근하고 커널 기능을 마음대로 쓸 수 있어, 2-2에서 본 **컨테이너 탈출로 가는 가장 짧은 길**이 됩니다.

> 🙋 (생각해 보기)
> `--pid=host` 컨테이너 안에서 `ps`로 1.1의 box-a, box-b nginx 프로세스가 보이는지 직접 확인해 보세요. 프로세스가 보이면 신호(`kill`)도 보낼 수 있습니다.

In [12]:
%%bash
docker run --rm --pid=host alpine:3.20 ps | grep "nginx: master" | grep -v grep

  936 root      0:00 nginx: master process nginx -g daemon off;
 1041 root      0:00 nginx: master p

rocess nginx -g daemon off;


### 3.3. 결과 정리

`[표 실습2-1]` 격리를 무너뜨리는 설정과 확인 결과

| 설정 | 사라지는 벽 | 확인한 결과 | 같은 뜻의 Pod 설정 |
|---|---|---|---|
| (기본값) root로 실행 | User 네임스페이스 없음 | 컨테이너의 root가 호스트에서도 root | `runAsUser: 0` |
| `-v /:/host` | Mount 네임스페이스 | 호스트에 root 소유 파일 생성 | `hostPath: /` |
| `--pid=host` | PID 네임스페이스 | 보이는 프로세스 2줄 → 수십 줄 | `hostPID: true` |
| `--privileged` | 장치·커널 권한 제한 | /dev 항목 15개 → 수백 개 | `privileged: true` |

맨 오른쪽 열은 다음 단계에서 Pod 설정 파일을 읽을 때 찾아야 할 항목입니다.

## 4. 배포 전에 설정 파일에서 찾기 (Trivy)

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | ✅ 완료 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | ✅ 완료 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | ✅ 완료 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | 예정 |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

3단계에서는 명령어 옵션으로 실행했지만, 실제 운영에서는 같은 설정이 쿠버네티스 Pod 설정 파일(YAML)에 들어갑니다. `k8s/bad-pod.yaml`에는 표 실습2-1의 위험 설정이 모두 들어 있습니다. **실제 클러스터에는 배포하지 마세요.**

### 4.1. 사람이 먼저 찾아보기

도구를 쓰기 전에 표 실습2-1의 오른쪽 열과 비교하며 위험한 줄을 직접 찾아봅니다.

In [13]:
print((K8S / "bad-pod.yaml").read_text())

# 2장 실습: 일부러 위험하게 만든 Pod (실제 클러스터에 배포하지 말 것)
# 아래 ①~⑦은 모두 "컨테이너 탈출"이나 "노드 장악"으로 이어질 수 있는 설정이다.
# good-pod.yaml과 한 줄씩 비교해 보고, kube-score·Trivy 같은 점검 도구가 무엇을 잡아내는지 확인한다.
apiVersion: v1
kind: Pod
metadata:
  name: bad-pod
spec:
  hostNetwork: true            # ① 노드의 네트워크를 그대로 사용
                               #    → 노드에서만 열려 있어야 할 포트(kubelet 등)에 컨테이너가 바로 접근할 수 있다
  hostPID: true                # ② 노드의 모든 프로세스가 보임
                               #    → 다른 컨테이너·노드 프로세스의 정보를 엿보거나 신호를 보낼 수 있다
  containers:
  - name: app
    image: nginx:latest        # ③ 버전이 고정되지 않은 이미지
                               #    → 배포할 때마다 다른 이미지가 올 수 있어, 무엇이 실행 중인지 추적하기 어렵다
    securityContext:
      privileged: true         # ④ 노드의 거의 모든 권한을 가짐
                               #    → 컨테이너 격리가 사실상 꺼진 상태. 가장 먼저 금지해야 할 설정
      runAsUser: 0             # ⑤ root로 실행
      allowPrivilegeEscalation: true   # 실행 중에 더 높은 권한을 얻는 것(setuid 등)을 허용
      capabilities:
        add: ["SYS_ADMIN"]     # ⑥ 강력한 커널 권한 추가
               

찾은 설정을 아래 목록에 적어 넣고 실행합니다. 4.2에서 Trivy 결과와 비교합니다. (예: `"hostPID: true"`)

In [14]:
my_findings = [
    # "hostPID: true",
    # 여기에 찾은 위험 설정을 한 줄씩 적습니다.
]
print(f"내가 찾은 위험 설정: {len(my_findings)}개")

내가 찾은 위험 설정: 0개


### 4.2. Trivy로 점검하기

Trivy는 클러스터 없이 **파일만으로** 설정을 점검합니다. 기본 출력은 규칙마다 해당 코드 줄까지 보여 주어 길기 때문에, 요약 줄과 규칙 줄만 골라 봅니다.

In [15]:
%%bash
trivy config k8s/bad-pod.yaml 2>/dev/null | grep -E '^(Tests|Failures|KSV)'

Tests: 98 (SUCCESSES: 73, FAILURES: 25)
Failures: 25 (UNKNOWN: 0, LOW: 12, MEDIUM: 6, HIGH: 7, CRITI

CAL: 0)
KSV-0001 (MEDIUM): Container 'app' of Pod 'bad-pod' should set 'securityContext.allowPrivile

geEscalation' to false
KSV-0003 (LOW): Container 'app' of Pod 'bad-pod' should add 'ALL' to 'securi

tyContext.capabilities.drop'
KSV-0004 (LOW): Container 'app' of 'pod' 'bad-pod' in 'default' namesp

ace should set securityContext.capabilities.drop
KSV-0005 (HIGH): Container 'app' of Pod 'bad-pod' 

...

curityContext.runAsUser should be set to a value greater than 0
KSV-0106 (LOW): container should dr

op all
KSV-0110 (LOW): pod bad-pod in default namespace should set metadata.namespace to a non-defa

ult namespace
KSV-0118 (HIGH): pod bad-pod in default namespace is using the default security conte

xt, which allows root privileges
KSV-0121 (HIGH): pod bad-pod in default namespace shouldn't have v

olumes set to {"/"}


규칙이 많으니 JSON으로 받아 파이썬으로 정리합니다. 아래 `trivy_scan()`은 4.4와 과제에서도 다시 씁니다.

> 📌 (참고) `trivy config -f json`의 결과에서 `Results[].Misconfigurations[]`가 어긴 규칙 목록이고, `MisconfSummary`에 통과·실패 개수가 있습니다.

In [16]:
SEVERITY_ORDER = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "UNKNOWN"]


def trivy_scan(path, show=True):
    """trivy config 결과를 심각도 순으로 정리해 표로 보여 주고, 어긴 규칙 목록을 돌려줍니다."""
    result = json.loads(sh(f"trivy config -q -f json {path}"))["Results"][0]
    summary = result["MisconfSummary"]
    fails = sorted(result.get("Misconfigurations", []),
                   key=lambda m: (SEVERITY_ORDER.index(m["Severity"]), m["ID"]))
    if show:
        counts = {s: sum(m["Severity"] == s for m in fails) for s in SEVERITY_ORDER[:4]}
        lines = [f"**{Path(path).name}** — 통과 {summary['Successes']}, 실패 {summary['Failures']} "
                 f"(HIGH {counts['HIGH']}, MEDIUM {counts['MEDIUM']}, LOW {counts['LOW']})", "",
                 "| 심각도 | 규칙 | 내용 | 해당 줄 |", "|---|---|---|---|"]
        for m in fails:
            start = m["CauseMetadata"].get("StartLine", "")
            lines.append(f"| {m['Severity']} | {m['ID']} | {m['Title']} | {start} |")
        display(Markdown("\n".join(lines)))
    return fails


bad = trivy_scan(K8S / "bad-pod.yaml")

**bad-pod.yaml** — 통과 73, 실패 25 (HIGH 7, MEDIUM 6, LOW 12)

| 심각도 | 규칙 | 내용 | 해당 줄 |
|---|---|---|---|
| HIGH | KSV-0005 | SYS_ADMIN capability added | 14 |
| HIGH | KSV-0009 | Access to host network | 8 |
| HIGH | KSV-0010 | Access to host PID | 8 |
| HIGH | KSV-0014 | Root file system is not read-only | 14 |
| HIGH | KSV-0017 | Privileged | 14 |
| HIGH | KSV-0118 | Default security context configured | 8 |
| HIGH | KSV-0121 | Kubernetes resource with disallowed volumes mounted | 8 |
| MEDIUM | KSV-0001 | Can elevate its own privileges | 14 |
| MEDIUM | KSV-0012 | Runs as root user | 14 |
| MEDIUM | KSV-0013 | Image tag ":latest" used | 14 |
| MEDIUM | KSV-0022 | Specific capabilities added | 14 |
| MEDIUM | KSV-0023 | hostPath volumes mounted | 8 |
| MEDIUM | KSV-0104 | Seccomp policies disabled | 14 |
| LOW | KSV-0003 | Default capabilities: some containers do not drop all | 14 |
| LOW | KSV-0004 | Default capabilities: some containers do not drop any | 14 |
| LOW | KSV-0011 | CPU not limited | 14 |
| LOW | KSV-0015 | CPU requests not specified | 14 |
| LOW | KSV-0016 | Memory requests not specified | 14 |
| LOW | KSV-0018 | Memory not limited | 14 |
| LOW | KSV-0020 | Runs with UID <= 10000 | 14 |
| LOW | KSV-0021 | Runs with GID <= 10000 | 14 |
| LOW | KSV-0030 | Runtime/Default Seccomp profile not set | 14 |
| LOW | KSV-0105 | Containers must not set runAsUser to 0 | 17 |
| LOW | KSV-0106 | Container capabilities must only include NET_BIND_SERVICE | 14 |
| LOW | KSV-0110 | Workloads in the default namespace | 6 |

HIGH 항목부터 보면 표 실습2-1의 설정이 모두 잡혔습니다.

`[표 실습2-2]` bad-pod.yaml에서 Trivy가 찾은 주요 항목

| 번호 | 설정 | Trivy 규칙 | 위험 |
|---|---|---|---|
| ① | `hostNetwork: true` | KSV-0009 (HIGH) | 노드의 네트워크를 그대로 써서, 노드 내부 서비스와 다른 Pod의 통신에 접근 |
| ② | `hostPID: true` | KSV-0010 (HIGH) | 3.2처럼 노드의 모든 프로세스가 보임 |
| ③ | `image: nginx:latest` | KSV-0013 (MEDIUM) | 언제 어떤 버전이 배포될지 모름 |
| ④ | `privileged: true` | KSV-0017 (HIGH) | 3.2처럼 격리가 사실상 해제됨 |
| ⑤ | `runAsUser: 0` | KSV-0012 (MEDIUM) | 2단계처럼 노드의 root와 같은 사용자로 실행 |
| ⑥ | `capabilities.add: SYS_ADMIN` | KSV-0005 (HIGH) | 마운트 같은 커널 수준의 강력한 작업 가능 |
| ⑦ | `hostPath: /` | KSV-0023 (MEDIUM), KSV-0121 (HIGH) | 3.1처럼 노드의 파일시스템 전체를 읽고 씀 |

④와 ⑦이 함께 있으면, 컨테이너에 들어온 공격자가 `/host`를 통해 노드의 파일을 바꾸고 노드에서 명령을 실행하는 **컨테이너 탈출**이 가능합니다.

### 4.3. 내가 찾은 것과 비교하기

4.1에서 적은 목록과 Trivy가 찾은 규칙의 Pod 설정 이름을 비교합니다. Trivy는 사람이 놓치기 쉬운 항목도 함께 찾습니다. 자원 한도 없음(KSV-0011, KSV-0018), 읽기 전용 파일시스템 미적용(KSV-0014), seccomp 미설정(KSV-0030) 같은 것입니다.

In [17]:
KEYWORDS = {   # 설정 이름 → 그 설정을 잡는 대표 규칙
    "hostNetwork": "KSV-0009", "hostPID": "KSV-0010", "latest": "KSV-0013",
    "privileged": "KSV-0017", "runAsUser": "KSV-0012", "SYS_ADMIN": "KSV-0005",
    "hostPath": "KSV-0023", "allowPrivilegeEscalation": "KSV-0001",
}
found_ids = {m["ID"] for m in bad}
for key, rule in KEYWORDS.items():
    mine = any(key in f for f in my_findings)
    print(f"{'✅' if mine else '⬜'} {key:26} Trivy {rule} {'검출' if rule in found_ids else '-'}")
print(f"\n사람이 놓치기 쉬운 항목까지 합쳐 Trivy가 찾은 규칙: {len(found_ids)}개")

⬜ hostNetwork                Trivy KSV-0009 검출
⬜ hostPID                    Trivy KSV-0010 검출
⬜ latest                     Trivy KSV-0013 검출
⬜ privileged                 Trivy KSV-0017 검출
⬜ runAsUser                  Trivy KSV-0012 검출
⬜ SYS_ADMIN                  Trivy KSV-0005 검출
⬜ hostPath                   Trivy KSV-0023 검출
⬜ allowPrivilegeEscalation   Trivy KSV-0001 검출

사람이 놓치기 쉬운 항목까지 합쳐 Trivy가 찾은 규칙: 25개


### 4.4. 안전하게 고친 `good-pod.yaml` 점검하기

`good-pod.yaml`은 bad-pod의 위험 설정을 하나씩 걷어 낸 파일입니다. 주석에 각 줄의 이유가 적혀 있습니다.

In [18]:
print((K8S / "good-pod.yaml").read_text())

# 2장 실습: bad-pod.yaml의 위험 설정을 하나씩 걷어 낸 안전한 Pod
# 원칙은 "필요한 것만 주고, 나머지는 기본으로 막는다(최소 권한)"이다.
apiVersion: v1
kind: Pod
metadata:
  name: good-pod
spec:
  automountServiceAccountToken: false   # 필요 없으면 API 토큰을 넣지 않음
                                        # (토큰이 있으면 컨테이너를 장악한 공격자가 Kubernetes API를 부를 수 있다)
  # hostNetwork·hostPID를 쓰지 않는다 → 기본값(false)으로 노드와 분리된다
  securityContext:                      # Pod 전체에 적용되는 보안 설정
    runAsNonRoot: true                  # root(UID 0)로는 시작하지 못하게 막는다
    runAsUser: 101                       # nginx 사용자
    seccompProfile:
      type: RuntimeDefault               # 위험한 시스템 호출 차단
  containers:
  - name: app
    image: nginxinc/nginx-unprivileged:1.27-alpine   # 버전 고정, 비root용 이미지
    ports:
    - containerPort: 8080               # 비root는 1024 미만 포트를 열 수 없어 8080을 쓴다
    securityContext:                    # 이 컨테이너에만 적용되는 보안 설정
      privileged: false                 # 특권 모드 끔
      allowPrivilegeEscalation: false   # 실행 중 권한 상승 금지
      readOnlyRootFilesys

In [19]:
good = trivy_scan(K8S / "good-pod.yaml")

**good-pod.yaml** — 통과 92, 실패 6 (HIGH 0, MEDIUM 1, LOW 5)

| 심각도 | 규칙 | 내용 | 해당 줄 |
|---|---|---|---|
| MEDIUM | KSV-0125 | Restrict container images to trusted registries | 17 |
| LOW | KSV-0015 | CPU requests not specified | 17 |
| LOW | KSV-0016 | Memory requests not specified | 17 |
| LOW | KSV-0020 | Runs with UID <= 10000 | 17 |
| LOW | KSV-0021 | Runs with GID <= 10000 | 17 |
| LOW | KSV-0110 | Workloads in the default namespace | 5 |

HIGH가 0건이 되었습니다. 남은 6건은 자원 요청값 미지정(KSV-0015, KSV-0016), UID·GID 번호가 10000 이하(KSV-0020, KSV-0021), default 네임스페이스 사용(KSV-0110), 신뢰한 레지스트리로 제한하지 않음(KSV-0125)입니다.

> 🙋 (생각해 보기)
> 이 가운데 무엇을 운영에서 받아들이고 무엇을 고칠지 판단해 보세요. 모든 경고로 배포를 막지 않고 **"막을 것과 알릴 것"을 나누는** 2-3의 원칙이 여기에 적용됩니다. 6.2 과제 1에서 직접 고쳐 봅니다.

## 5. (선택) 클러스터가 위험한 Pod를 거부하게 만들기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | ✅ 완료 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | ✅ 완료 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | ✅ 완료 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | ✅ 완료 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 체크포인트와 과제 | 예정 |
| 7 | 정리 | 예정 |

4단계가 위험한 설정을 "찾는" 것이라면, 5단계는 사람이 실수로 넣어도 클러스터가 배포 단계에서 "막는" 것입니다. 쿠버네티스 클러스터가 없다면 Docker 위에서 도는 실습용 클러스터 **kind**로 만듭니다.

### 5.1. kind와 kubectl 설치

0단계에서 `kind`, `kubectl`이 ✅로 나왔다면 이 셀은 건너뜁니다. 내려받은 파일은 공식 체크섬으로 확인합니다(`OK`가 두 번 나와야 합니다).

In [20]:
%%bash
mkdir -p ~/bin && cd ~/bin
curl -fsSLo kind https://github.com/kubernetes-sigs/kind/releases/download/v0.24.0/kind-linux-amd64
curl -fsSLo kind.sha256 https://github.com/kubernetes-sigs/kind/releases/download/v0.24.0/kind-linux-amd64.sha256sum
echo "$(awk '{print $1}' kind.sha256)  kind" | sha256sum -c
curl -fsSLo kubectl https://dl.k8s.io/release/v1.31.1/bin/linux/amd64/kubectl
echo "$(curl -fsSL https://dl.k8s.io/release/v1.31.1/bin/linux/amd64/kubectl.sha256)  kubectl" | sha256sum -c
chmod +x kind kubectl

kind: OK


kubectl: OK


### 5.2. 실습용 클러스터 만들기

처음에는 노드 이미지를 받느라 몇 분 걸립니다. 같은 이름의 클러스터가 이미 있으면 그대로 씁니다.

In [21]:
%%bash
kind get clusters 2>/dev/null | grep -qx pod-lab \
  || kind create cluster --name pod-lab --image kindest/node:v1.31.0 2>&1 | grep -v '^ •'
kubectl config use-context kind-pod-lab
kubectl wait --for=condition=Ready node --all --timeout=180s >/dev/null   # 노드가 준비될 때까지 기다림
kubectl get nodes

Creating cluster "pod-lab" ...
 ✓ Ensuring node image (kindest/node:v1.31.0) 🖼
 ✓ Preparing n

odes 📦 
 ✓ Writing configuration 📜
 ✓ Starting control-plane 🕹️
 ✓ Installing CNI 

🔌
 ✓ Installing StorageClass 💾
Set kubectl context to "kind-pod-lab"
You can now use your clu

ster with:

kubectl cluster-info --context kind-pod-lab

Thanks for using kind! 😊


Switched to context "kind-pod-lab".


NAME                    STATUS   ROLES           AGE   VERSION
pod-lab-control-plane   Ready    cont

rol-plane   21s   v1.31.0


### 5.3. 네임스페이스에 restricted 기준 걸기

쿠버네티스에는 **Pod Security Admission**이라는 검사 기능이 기본으로 들어 있습니다. 네임스페이스에 라벨 하나를 붙이면, 그 네임스페이스에 들어오는 모든 Pod를 쿠버네티스 공식 기준인 Pod Security Standards로 검사합니다.

In [22]:
%%bash
kubectl create namespace secure-lab --dry-run=client -o yaml | kubectl apply -f -
kubectl label namespace secure-lab pod-security.kubernetes.io/enforce=restricted --overwrite
until kubectl get serviceaccount default -n secure-lab >/dev/null 2>&1; do sleep 1; done   # 기본 서비스 계정이 생길 때까지 기다림
kubectl get namespace secure-lab --show-labels

namespace/secure-lab created


namespace/secure-lab labeled


NAME         STATUS   AGE   LABELS
secure-lab   Active   1s    kubernetes.io/metadata.name=secure-la

b,pod-security.kubernetes.io/enforce=restricted


> 📌 (참고) 네임스페이스를 만든 직후에는 기본 서비스 계정(`default`)이 몇 초 늦게 생깁니다. 그 전에 Pod를 배포하면 `serviceaccount "default" not found`로 거부되어 Pod 보안 검사 결과를 볼 수 없으므로, 위 셀은 서비스 계정이 생길 때까지 기다립니다.

### 5.4. bad-pod와 good-pod 배포해 보기

In [23]:
%%bash
kubectl apply -f k8s/bad-pod.yaml -n secure-lab 2>&1 | fold -w 120 || true
echo
kubectl apply -f k8s/good-pod.yaml -n secure-lab
kubectl wait --for=condition=Ready pod/good-pod -n secure-lab --timeout=120s >/dev/null
kubectl get pods -n secure-lab

Error from server (Forbidden): error when creating "k8s/bad-pod.yaml": pods "bad-pod" is forbidden: 

violates PodSecurity
 "restricted:latest": host namespaces (hostNetwork=true, hostPID=true), privile

ged (container "app" must not set securit
yContext.privileged=true), allowPrivilegeEscalation != fal

se (container "app" must set securityContext.allowPrivilegeEsc
alation=false), unrestricted capabili

ties (container "app" must set securityContext.capabilities.drop=["ALL"]; container
 "app" must not 

...

curityContext.runAsNonRoo
t=true), runAsUser=0 (container "app" must not set runAsUser=0), seccompPr

ofile (pod or container "app" must set securit
yContext.seccompProfile.type to "RuntimeDefault" or "

Localhost")


pod/good-pod created


NAME       READY   STATUS    RESTARTS   AGE
good-pod   1/1     Running   0          7s


bad-pod는 `Forbidden`으로 거부되고, good-pod만 `Running`이 되었습니다.

라벨 `pod-security.kubernetes.io/enforce=restricted`는 두 가지를 정합니다. 앞의 **enforce**는 기준을 어긴 Pod를 어떻게 다룰지(모드)이고, 뒤의 **restricted**는 어떤 기준으로 검사할지(단계)입니다.

`[표 실습2-3]` Pod Security Admission의 모드와 단계

| 구분 | 값 | 뜻 |
|---|---|---|
| 모드 | `enforce` | 기준을 어긴 Pod의 생성을 거부 |
| | `warn` | 생성은 허용하고, 명령을 실행한 사람에게 경고만 표시 |
| | `audit` | 생성은 허용하고, 클러스터 감사 로그에만 기록 |
| 단계 | `privileged` | 제한 없음 (시스템 관리용 Pod 전용) |
| | `baseline` | 알려진 권한 상승 경로를 막는 최소 기준 (호스트 네임스페이스, privileged, hostPath 등 금지) |
| | `restricted` | baseline에 더해 root 실행 금지, 권한 상승 금지, 커널 권한 전부 제거 같은 강화 기준까지 요구 |

### 5.5. 거부 메시지를 규칙으로 나눠 읽기

거부 메시지는 한 줄로 길게 나옵니다. 파이썬으로 항목별로 나눠 봅니다. `--dry-run=server`는 클러스터의 검사는 받되 실제로 만들지는 않는 옵션입니다.

In [24]:
import re

msg = subprocess.run("kubectl apply --dry-run=server -f k8s/bad-pod.yaml -n secure-lab",
                     shell=True, capture_output=True, text=True).stderr
violations = msg.split('"restricted:latest": ', 1)[1].strip()
for i, item in enumerate(re.split(r"\), (?=[a-zA-Z])", violations), 1):
    print(f"{i}. {item.rstrip(')')})")

1. host namespaces (hostNetwork=true, hostPID=true)
2. privileged (container "app" must not set securityContext.privileged=true)
3. allowPrivilegeEscalation != false (container "app" must set securityContext.allowPrivilegeEscalation=false)
4. unrestricted capabilities (container "app" must set securityContext.capabilities.drop=["ALL"]; container "app" must not include "SYS_ADMIN" in securityContext.capabilities.add)
5. restricted volume types (volume "host-root" uses restricted volume type "hostPath")
6. runAsNonRoot != true (pod or container "app" must set securityContext.runAsNonRoot=true)
7. runAsUser=0 (container "app" must not set runAsUser=0)
8. seccompProfile (pod or container "app" must set securityContext.seccompProfile.type to "RuntimeDefault" or "Localhost")


`[표 실습2-4]` bad-pod가 어긴 restricted 규칙

| 거부 메시지의 항목 | 규칙 | 단계 | bad-pod의 설정 |
|---|---|---|---|
| host namespaces | 노드의 네트워크·프로세스·IPC 공유 금지 | baseline | `hostNetwork: true`, `hostPID: true` |
| privileged | 특권 컨테이너 금지 | baseline | `privileged: true` |
| allowPrivilegeEscalation != false | 권한 상승을 명시적으로 막아야 함 | restricted | `allowPrivilegeEscalation: true` |
| unrestricted capabilities | 커널 권한을 모두 버리고(`drop: ALL`) `NET_BIND_SERVICE`만 추가 가능 | restricted | `SYS_ADMIN` 추가, `drop` 없음 |
| restricted volume types | configMap, emptyDir, secret 같은 허용된 볼륨만 사용 (hostPath 금지) | restricted | `hostPath: /` |
| runAsNonRoot != true, runAsUser=0 | root로 실행 금지 | restricted | `runAsUser: 0` |
| seccompProfile | seccomp 프로필을 RuntimeDefault나 Localhost로 지정 | restricted | 지정 없음 |

> 📌 (참고) Pod Security Standards는 **실행 권한만** 봅니다. 4단계에서 Trivy가 찾은 `nginx:latest`(KSV-0013)나 신뢰하지 않은 레지스트리(KSV-0125)는 검사하지 않으므로, 이런 규칙까지 배포 단계에서 강제하려면 Kyverno나 OPA Gatekeeper 같은 별도 정책 도구가 필요합니다. 운영 중인 클러스터에 처음 적용할 때는 `warn`이나 `audit`으로 먼저 걸리는 Pod를 확인한 뒤 `enforce`로 바꾸는 순서가 안전합니다(과제 2).

## 6. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | ✅ 완료 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | ✅ 완료 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | ✅ 완료 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | ✅ 완료 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 정리 | 예정 |

### 6.1. 체크포인트

- [ ] 컨테이너 안의 PID 1이 호스트에서는 평범한 프로세스라는 것을 확인했나요?
- [ ] 컨테이너 안의 root가 호스트에서도 root로 실행되고 있다는 것을 확인했나요?
- [ ] `-v /:/host`, `--pid=host`, `--privileged`가 각각 어떤 벽을 없애는지 설명할 수 있나요?
- [ ] 같은 위험 설정을 Pod 설정 파일에서 배포 전에 찾고 고쳤나요?

### 6.2. 과제 1: good-pod의 남은 경고 줄이기

4.4에서 good-pod에 남은 6건 가운데 **고칠 것을 골라** 고친 `my-pod.yaml`을 만듭니다. 아래 셀은 good-pod.yaml을 그대로 복사해 둔 것입니다. `# TODO` 주석을 참고해 고친 뒤 실행하면, 다음 셀이 Trivy로 다시 점검합니다.

- KSV-0015, KSV-0016: `resources.requests`(cpu, memory)를 추가합니다.
- KSV-0110: `metadata.namespace`를 `default`가 아닌 값으로 지정합니다.
- KSV-0020, KSV-0021: Pod의 `securityContext`에 10000보다 큰 `runAsUser`, `runAsGroup`을 지정합니다.
- KSV-0125: 레지스트리를 명시하면 해결될까요? 직접 확인해 보세요.

> 📌 (참고) `%%writefile`은 셀 내용을 그대로 파일로 저장하는 명령입니다. 고친 파일은 `k8s/my-pod.yaml`에 생깁니다.

In [25]:
%%writefile k8s/my-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: my-pod
  # TODO(KSV-0110): namespace를 지정합니다.
spec:
  automountServiceAccountToken: false
  securityContext:
    runAsNonRoot: true
    runAsUser: 101                       # TODO(KSV-0020): 10000보다 큰 값으로
    # TODO(KSV-0021): runAsGroup을 추가합니다.
    seccompProfile:
      type: RuntimeDefault
  containers:
  - name: app
    image: nginxinc/nginx-unprivileged:1.27-alpine
    ports:
    - containerPort: 8080
    securityContext:
      privileged: false
      allowPrivilegeEscalation: false
      readOnlyRootFilesystem: true
      capabilities:
        drop: ["ALL"]
    resources:
      # TODO(KSV-0015, KSV-0016): requests를 추가합니다.
      limits:
        cpu: "250m"
        memory: "128Mi"
    volumeMounts:
    - name: tmp
      mountPath: /tmp
  volumes:
  - name: tmp
    emptyDir: {}

Writing k8s/my-pod.yaml


In [26]:
mine = trivy_scan(K8S / "my-pod.yaml")
left = {m["ID"] for m in mine}
solved = {m["ID"] for m in good} - left
print(f"good-pod 대비 해결: {sorted(solved) or '없음'}")
print(f"남은 규칙: {sorted(left) or '없음'}")

**my-pod.yaml** — 통과 92, 실패 6 (HIGH 0, MEDIUM 1, LOW 5)

| 심각도 | 규칙 | 내용 | 해당 줄 |
|---|---|---|---|
| MEDIUM | KSV-0125 | Restrict container images to trusted registries | 15 |
| LOW | KSV-0015 | CPU requests not specified | 15 |
| LOW | KSV-0016 | Memory requests not specified | 15 |
| LOW | KSV-0020 | Runs with UID <= 10000 | 15 |
| LOW | KSV-0021 | Runs with GID <= 10000 | 15 |
| LOW | KSV-0110 | Workloads in the default namespace | 3 |

good-pod 대비 해결: 없음
남은 규칙: ['KSV-0015', 'KSV-0016', 'KSV-0020', 'KSV-0021', 'KSV-0110', 'KSV-0125']


<details>
<summary>💡 힌트 보기</summary>

```yaml
metadata:
  name: my-pod
  namespace: secure-lab
spec:
  securityContext:
    runAsNonRoot: true
    runAsUser: 10101
    runAsGroup: 10101
  ...
    resources:
      requests:
        cpu: "100m"
        memory: "64Mi"
      limits:
        ...
```

이렇게 고치면 6건이 KSV-0125 1건으로 줄어듭니다. 이미지 이름 앞에 `docker.io/`를 붙여도 KSV-0125는 남습니다. Trivy의 기본 신뢰 레지스트리 목록에 Docker Hub가 없기 때문입니다.

남은 규칙이 있다면 "왜 남겨 둘지"를 한 줄로 적어 보세요. 예를 들어 사내 레지스트리가 없는 실습 환경에서는 KSV-0125를 고칠 수 없으므로, 운영에서는 사내 레지스트리로 이미지를 옮긴 뒤 이 규칙을 **막을 것**으로 바꿉니다.
</details>

> 🙋 (생각해 보기)
> UID를 101에서 10101로 바꾸면 nginx가 실제로 뜰까요? Trivy는 설정 파일만 보므로 이 질문에 답하지 못합니다. 5단계 클러스터가 있다면 `kubectl apply -f k8s/my-pod.yaml`로 직접 확인해 보세요. 점검 도구의 "통과"와 서비스의 "정상 동작"은 따로 확인해야 합니다.

In [27]:
%%bash
# (선택) 5단계 클러스터가 있다면 실제로 떠 보는지 확인합니다.
kubectl apply -f k8s/my-pod.yaml -n secure-lab && sleep 15 && kubectl get pod my-pod -n secure-lab

pod/my-pod created


NAME     READY   STATUS    RESTARTS   AGE
my-pod   1/1     Running   0          15s


### 6.3. 과제 2: enforce 대신 warn 모드로 바꿔 보기 (5단계를 한 경우)

운영 중인 클러스터에 처음 기준을 걸 때는 `warn` 모드로 먼저 어떤 Pod가 걸리는지 봅니다. 아래 셀은 `warn-lab` 네임스페이스를 **warn 모드**로 만들고 bad-pod를 배포합니다.

1. 셀을 실행하고, 5.4의 enforce 결과와 무엇이 다른지 적습니다.
2. 배포된 bad-pod가 실제로 실행 중인지 확인합니다. 이것이 왜 위험한지 3단계 결과와 연결해 설명합니다.

In [28]:
%%bash
kubectl create namespace warn-lab --dry-run=client -o yaml | kubectl apply -f -
kubectl label namespace warn-lab pod-security.kubernetes.io/warn=restricted --overwrite
until kubectl get serviceaccount default -n warn-lab >/dev/null 2>&1; do sleep 1; done
kubectl apply -f k8s/bad-pod.yaml -n warn-lab 2>&1 | fold -w 120
sleep 10
kubectl get pods -n warn-lab
kubectl delete namespace warn-lab --wait=false   # 위험한 Pod이므로 확인 후 바로 지웁니다

namespace/warn-lab created


namespace/warn-lab labeled


rue), privileged (co
ntainer "app" must not set securityContext.privileged=true), allowPrivilegeEsca

lation != false (container "app" must set
 securityContext.allowPrivilegeEscalation=false), unrestri

cted capabilities (container "app" must set securityContext.ca
pabilities.drop=["ALL"]; container "a

...

" mu
st set securityContext.runAsNonRoot=true), runAsUser=0 (container "app" must not set runAsUser=

0), seccompProfile (pod o
r container "app" must set securityContext.seccompProfile.type to "Runtime

Default" or "Localhost")
pod/bad-pod created


NAME      READY   STATUS    RESTARTS   AGE
bad-pod   1/1     Running   0          11s


namespace "warn-lab" deleted


✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

- enforce와 warn의 차이:
- bad-pod가 실행 중일 때의 위험:

### 6.4. 과제 3: 장애 대응 요청에 답하기 (서술)

새벽 장애 중에 개발팀이 "권한 문제 같으니 운영 Pod에 `privileged: true`를 잠깐 넣어도 되냐"고 묻습니다. 이 실습의 결과를 근거로, 보안 담당자로서 **허용 조건이나 대안**을 세 줄 이내로 답해 보세요.

> 💡 참고할 실습 결과: 3.2의 `/dev` 항목 수, 4.2의 KSV-0017, 5.4의 거부 메시지, 그리고 `capabilities.add`로 필요한 권한 하나만 주는 방법

✍️ **답 적기**

1.
2.
3. 

## 7. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비: 도구와 실습 파일 확인 | ✅ 완료 |
| 1 | 컨테이너 두 개 띄우고 격리 확인하기 | ✅ 완료 |
| 2 | 컨테이너 안의 root는 호스트에서 누구인가 | ✅ 완료 |
| 3 | 설정 한 줄로 벽 무너뜨리기 | ✅ 완료 |
| 4 | 배포 전에 설정 파일에서 찾기 (Trivy) | ✅ 완료 |
| 5 | (선택) 클러스터가 위험한 Pod를 거부하게 만들기 | ✅ 완료 |
| 6 | 체크포인트와 과제 | ✅ 완료 |
| 7 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

실습에서 만든 컨테이너, 파일, 이미지, 클러스터를 지웁니다.

> 📌 (참고) 3.1에서 만든 `/tmp/escaped-from-container`는 root 소유라 일반 사용자로는 지울 수 없습니다. 노트북에서는 `sudo` 암호를 입력할 수 없으므로, 같은 방법(컨테이너의 root + 호스트 경로 마운트)으로 지웁니다. 3.1의 위험이 그대로 드러나는 장면이기도 합니다.

In [29]:
%%bash
docker rm -f box-a box-b
docker run --rm -v /tmp:/host-tmp alpine:3.20 rm -f /host-tmp/escaped-from-container
ls /tmp/escaped-from-container 2>/dev/null || echo "/tmp/escaped-from-container 삭제됨"
docker rmi nginx:1.27-alpine alpine:3.20
rm -f k8s/my-pod.yaml

box-a
box-b


/tmp/escaped-from-container 삭제됨


Untagged: nginx:1.27-alpine
Deleted: sha256:65645c7bb6a0661892a8b03b89d0743208a18dd2f3f17a54ef4b76fb

8e2f2a10


Untagged: alpine:3.20
Deleted: sha256:d9e853e87e55526f6b2917df91a2115c36dd7c696a35be12163d44e6e2a4b6

bc


In [30]:
%%bash
# 5단계를 했다면 실행합니다.
kind delete cluster --name pod-lab

Deleting cluster "pod-lab" ...


Deleted nodes: ["pod-lab-control-plane"]


---
> 이미지 안에 든 패키지의 취약점(CVE)은 8장 실습 1에서 직접 만든 이미지로 스캔하고, 보안 Gate로 배포 여부를 판정합니다.